# Visuals
- Visualize Token Merging within SReT per-stage for a sample image.

In [ ]:
import torch
import tome
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from PIL import Image
from functools import partial
import SReT_ToMe

# dataset transforms
transform_list = [
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224)
]
transform_vis  = transforms.Compose(transform_list)
transform_norm = transforms.Compose(transform_list + [
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# load image
img = Image.open("images/husky.png").convert("RGB")
img_vis = transform_vis(img)
img_norm = transform_norm(img)
img_batch = img_norm[None, ...].cuda()

# define hierarchical structure of SReT
stage_configs = [
    {"stage_idx": 0, "patch_size": 8,  "grid": "28x28", "name": "Stage 1 (High Res)"},
    {"stage_idx": 1, "patch_size": 16, "grid": "14x14", "name": "Stage 2 (Mid Res)"},
    {"stage_idx": 2, "patch_size": 32, "grid": "7x7",   "name": "Stage 3 (Deep Res)"}
]

for config in stage_configs:
    idx = config["stage_idx"]

    # initialize model
    model = SReT_ToMe.SReT_T_distill(pretrained=False, initial_r_ratio=0.3, alpha=0.1)
    checkpoint = torch.load('weights/SReT_T_distill.pth', map_location='cpu')
    model.load_state_dict(checkpoint['model'])
    model = model.cuda().eval()

    # inject trace_source to target stage
    model.transformers[idx].forward = partial(model.transformers[idx].forward, trace_source=True)

    # run inference
    with torch.no_grad():
        _ = model(img_batch)

    # extract trace 
    source = model.transformers[idx]._tome_info["source"]
    
    print(f"\n--- {config['name']} ---")
    print(f"Starting Tokens: {source.shape[2]} ({config['grid']})")
    print(f"Remaining Tokens after ToMe: {source.shape[1]}")
    
    # generate the visualization
    result = tome.make_visualization(img_vis, source, patch_size=config["patch_size"], class_token=False)
    filename = f"images/stage_{idx + 1}_map.png"
    result.save(filename)
    print(f"Saved visualization to {filename}")



--- Stage 1 (High Res) ---
Starting Tokens: 784 (28x28)
Remaining Tokens after ToMe: 512
Saved visualization to images/stage_1_map.png

--- Stage 2 (Mid Res) ---
Starting Tokens: 196 (14x14)
Remaining Tokens after ToMe: 128
Saved visualization to images/stage_2_map.png

--- Stage 3 (Deep Res) ---
Starting Tokens: 49 (7x7)
Remaining Tokens after ToMe: 34
Saved visualization to images/stage_3_map.png
